In [2]:
!pip install -qU langchain-ollama langchain-chroma langchain-text-splitters langchain pypdf

In [3]:
import sys
!{sys.executable} -m pip install -qU langchain

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\hp\AppData\Local\Temp\ipykernel_7732\3577633446.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
PDF_PATH = ".././NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf" 
PERSIST_DIRECTORY = "./local_chroma_db"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2"

In [3]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

In [ ]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY,
    collection_name="rag_collection"
)

In [11]:
from langchain_core.output_parsers import StrOutputParser

# --- 4. Define Generation Chain ---
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
llm = OllamaLLM(model=LLM_MODEL, temperature=0.1)

system_prompt = (
    "Use the following context to answer the question. "
    "If the context does not contain the answer, say 'I don't have enough context.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

# Only pipe the prompt, model, and string parser
generation_chain = prompt | llm | StrOutputParser()


# --- 5. Step-by-Step Query Execution ---
print("RAG System is ready! (Type 'exit' or 'quit' to stop)\n")

while True:
    # This creates a text box in Jupyter for you to type your query
    query = input("Enter your question: ")
    
    # Check if the user wants to stop
    if query.lower() in ['exit', 'quit']:
        print("Exiting chat.")
        break
        
    print(f"\nThinking about: '{query}'...")
    
    # Step A: Retrieve the raw document chunks explicitly
    retrieved_docs = retriever.invoke(query)

    # Step B: Format the chunks into a single text block
    formatted_context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Step C: Feed the formatted context and query to the LLM
    answer = generation_chain.invoke({
        "context": formatted_context,
        "question": query
    })

    print("\n--- Answer ---")
    print(answer)

    print("\n--- Source Documents ---")
    # Print the metadata from the retrieved chunks to verify attribution
    for doc in retrieved_docs:
        print(f"- Source: {doc.metadata.get('source', 'Unknown')}, Page: {doc.metadata.get('page', 'Unknown')}")
    
    print("\n" + "="*50 + "\n") # Divider for the next question

RAG System is ready! (Type 'exit' or 'quit' to stop)



Enter your question:  What does the text mention about taccel?



Thinking about: 'What does the text mention about taccel?'...

--- Answer ---
According to the text, Taccel is a platform that provides high-performance GPU simulation for vision-based tactile sensing. It allows for seamless integration of diverse robotic platforms and sensor configurations, and provides a user-friendly API for easy loading and configuration of robots. Taccel's utility is validated through three fundamental tactile-informed robotic tasks, including object classification and grasping experiments. The platform uses a unified ABD-IPC (Augmented Body Dynamics and Inverse Kinematics) algorithm to balance computational efficiency with physical accuracy, and can simulate both rigid and soft objects.

--- Source Documents ---
- Source: .././NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 24
- Source: .././NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation

Enter your question:  exit


Exiting chat.
